# Logistic Regression in Python

---

## 1. Introduction

This notebook demonstrates how to fit logistic regression models for **binary** outcome variables—variables that can take only two distinct values. As an example, we will use the variable `smoked`, which we will derive from the `SMQ020` variable in the NHANES dataset.

The `smoked` variable indicates whether a person has smoked at least 100 cigarettes in their lifetime (a common definition of having a "smoking history"). We will recode this variable so that individuals with a smoking history are coded as 1 (smoker) and those without as 0 (non-smoker). Rare or ambiguous responses (such as *don't know* or *refused to answer*) will be treated as missing values.

In our analysis, we will explore how plausible predictors—such as age, gender, and educational attainment—are associated with smoking history using logistic regression models.

---

## 2. Import Libraries, Load and Prepare the Data
First, we import the standard libraries for data manipulation, visualization, and statistical modeling (e.g., pandas, numpy, matplotlib/seaborn, and statsmodels). 

Then we load the NHANES 2015–2016 data. NHANES includes multiple waves; this notebook uses only the 2015–2016 wave.

As with most datasets, NHANES contains missing values. For simplicity we perform a **complete‑case analysis**: we  drop observations with missing values in any of the key variables used in this notebook. 

> **Note:** Complete‑case analysis reduces the effective sample size and can bias estimates and standard errors unless the missingness mechanism is missing completely at random (MCAR). In practice you should consider alternatives such as single or multiple imputation, or other methods that explicitly model missingness.

In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

# Read the 2015-2016 wave of NHANES data
nhanes_df = pd.read_csv("./data/nhanes_2015_2016.csv")

# Select columns of interest and drop observations with missing values
nhanes_df["smoked"] = nhanes_df["SMQ020"].replace({2: 0, 7: np.nan, 9: np.nan})
cols = ["DMDEDUC2", "RIAGENDR", "RIDAGEYR", "smoked"]
nhanes_df = nhanes_df[cols].dropna().reset_index(drop=True)

# Rename columns
nhanes_df = nhanes_df.rename(columns={"RIDAGEYR" : "age", "RIAGENDR": "gender", "DMDEDUC2": "education"})

# Add a `gender_labeled` column
nhanes_df["gender_labeled"] = nhanes_df["gender"].replace({1: "Male", 2:"Female"})

# Convert `education` and `smoked` columns' data types to category
nhanes_df["education"] = nhanes_df["education"].astype("int").astype("category")
nhanes_df["smoked"] = nhanes_df["smoked"].astype("int").astype("category")

# Print the first 5 rows
nhanes_df.head()

,education,gender,age,smoked,gender_labeled
0,5,1,62,1,Male
1,3,1,53,1,Male
2,3,1,78,1,Male
3,5,2,56,0,Female
4,4,2,42,0,Female



---

## 3. Odds and Log Odds

Logistic regression models the **odds** of an event occurring, rather than modeling the probability directly. Recall that if an event has probability $p$, the odds of the event are defined as $p / (1 - p)$. In other words, the odds represent the ratio of the probability that the event occurs to the probability that it does not occur. For example, if the probability is $1/2$, then the odds are $1$.

This transformation allows us to work on a different scale, which has mathematical advantages in regression modeling.

To illustrate, let’s start by examining the odds of smoking separately for women and men.

In [18]:
# Create a contingency table (counts) of gender vs. smoking status
counts = pd.crosstab(nhanes_df["gender_labeled"], nhanes_df["smoked"])

# Convert the counts to proportions within each gender (row-wise proportions)
proportions = counts.div(counts.sum(axis=1), axis=0)

# Calculate the odds of smoking for each gender
proportions["odds"] = proportions[1] / proportions[0]

# Display the table with proportions and odds
proportions

smoked,0,1,odds
gender_labeled,,,
Female,0.684821,0.315179,0.460236
Male,0.467914,0.532086,1.137143


We see that the probabilty that a woman has ever smoked is substantially lower than the probability that a man has ever smoked (32% vs 53%). This is reflected in the odds for a woman smoking being much less than 1 (around 0.47), while the odds for a man smoking is around 1.14.

It is common to work with **odds ratios** when comparing two groups. This is simply the odds for one group divided by the odds for the other group. The odds ratio for smoking, comparing males to females, is around 2.4. In other words, a man has around 2.47 times greater odds of smoking than a woman (in the population represented by these data).

In [20]:
print(proportions["odds"]["Male"] / proportions["odds"]["Female"])

2.470781971651537
